In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 현재 미리 보기 릴리스 상태이며 변경될 수 있습니다.

## 개요
IBM&reg; Circuit function은 [추상 PUB](./primitive-input-output)을 입력으로 받아, 완화된 기댓값을 출력으로 반환합니다. 이 Circuit function에는 연구자들이 알고리즘 및 애플리케이션 탐구에 집중할 수 있도록 자동화되고 맞춤화된 파이프라인이 포함되어 있습니다.

## 설명
PUB를 제출하면, 추상 Circuit과 관측값이 자동으로 트랜스파일되고, 하드웨어에서 실행되며, 완화된 기댓값을 반환하기 위해 후처리됩니다. 이를 위해 다음 도구들을 결합합니다.

- [Qiskit Transpiler Service](./qiskit-transpiler-service): AI 기반 및 휴리스틱 트랜스파일 패스의 자동 선택을 통해 추상 Circuit을 하드웨어 최적화된 ISA Circuit으로 변환합니다.
- [유틸리티 규모 계산에 필요한 오류 억제 및 완화](./error-mitigation-and-suppression-techniques): 측정 및 Gate 트위를링, 동적 디커플링, Twirled Readout Error eXtinction(TREX), Zero-Noise Extrapolation(ZNE), Probabilistic Error Amplification(PEA)이 포함됩니다.
- [Qiskit Runtime Estimator](./get-started-with-primitives): ISA PUB를 하드웨어에서 실행하고 완화된 기댓값을 반환합니다.

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## 시작하기
[API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고 다음과 같이 Qiskit Function을 선택하세요. (이 코드 스니펫은 이미 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)해 두었다고 가정합니다.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## 예제
시작하려면 다음 기본 예제를 사용해 보세요.

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Qiskit Function 워크로드의 [상태](/guides/functions#check-job-status)를 확인하거나 다음과 같이 [결과](/guides/functions#retrieve-results)를 반환하세요.

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


결과는 [Estimator 결과](/guides/primitive-input-output#estimator-output)와 동일한 형식입니다.

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## 입력
IBM Circuit function이 허용하는 모든 입력 매개변수는 아래 표를 참조하세요. 이어지는 [옵션](#options) 섹션에서 사용 가능한 `options`에 대해 더 자세히 설명합니다.

| 이름 | 타입 | 설명 | 필수 여부 | 기본값 | 예시 |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | 쿼리를 실행할 Backend의 이름입니다.                                                                                                                                                                                              | 예      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | `(circuit, observables)` 또는 `(circuit, observables, parameter_values)`와 같은 튜플 형태의 추상 PUB 유사(primitive unified bloc) 객체의 이터러블입니다. 자세한 내용은 [PUB 개요](/guides/primitive-input-output#overview-of-pubs)를 참조하세요. Circuit은 추상(비ISA)일 수 있습니다. | 예      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | 입력 옵션입니다. 자세한 내용은 **옵션** 섹션을 참조하세요.                                                                                                                                                                                | 아니요       | 자세한 내용은 **옵션** 섹션을 참조하세요.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | 사용할 인스턴스의 클라우드 리소스 이름입니다.                                                                                                                                                                                        | 아니요       | 계정이 여러 인스턴스에 접근 가능한 경우 무작위로 하나가 선택됩니다. | `CRN`                   |

### 옵션
#### 구조
Qiskit Runtime primitives와 유사하게, IBM Circuit function의 옵션은 중첩 딕셔너리로 지정할 수 있습니다. ``optimization_level`` 및 ``mitigation_level``과 같이 자주 사용되는 옵션은 첫 번째 레벨에 있습니다. 그 외 고급 옵션들은 ``resilience``와 같이 서로 다른 카테고리로 그룹화되어 있습니다.

#### 기본값
옵션에 값을 지정하지 않으면 서비스에서 지정한 기본값이 사용됩니다.

#### 완화 수준
IBM Circuit function은 `mitigation_level`도 지원합니다. 완화 수준은 작업에 적용할 오류 억제 및 완화의 양을 지정합니다. 수준이 높을수록 더 정확한 결과를 생성하지만 처리 시간이 더 오래 걸립니다. 오류 감소의 정도는 적용되는 방법에 따라 달라집니다. 완화 수준은 상세한 오류 완화 및 억제 방법의 선택을 추상화하여 사용자가 애플리케이션에 적합한 비용/정확도 절충을 판단할 수 있게 합니다. 아래 표는 각 수준에 해당하는 방법을 보여줍니다.

> **Note:** 이름이 비슷하지만, `mitigation_level`은 Estimator의 `resilience_level`이 사용하는 기법과는 다른 기법을 적용합니다.

Qiskit Runtime Estimator의 ``resilience_level``과 유사하게, ``mitigation_level``은 기본 큐레이션 옵션 세트를 지정합니다. 완화 수준에 추가로 수동 지정하는 옵션은 완화 수준이 정의하는 기본 옵션 세트 위에 적용됩니다. 따라서 원칙적으로 완화 수준을 1로 설정하더라도 측정 완화를 끌 수 있지만, 이는 권장되지 않습니다.

| **완화 수준** | **기법** |
|:-:|:-:|
| 1 [기본값] | 동적 디커플링 + 측정 트위를링 + TREX  |
| 2 | 수준 1 + Gate 트위를링 + Gate 폴딩을 통한 ZNE |
| 3 | 수준 1 + Gate 트위를링 + PEA를 통한 ZNE |

다음 예제는 완화 수준 설정 방법을 보여줍니다.

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### 사용 가능한 모든 옵션
`mitigation_level` 외에도, IBM Circuit function은 비용/정확도 절충을 세밀하게 조정할 수 있는 일부 고급 옵션을 제공합니다. 다음 표는 사용 가능한 모든 옵션을 보여줍니다.

| 옵션 | 하위 옵션 | 하위-하위 옵션 | 설명 | 선택 가능한 값 | 기본값 |
|-|-|-|-|-|-|
| default_precision |  |  | 정밀도를 지정하지 않은 PUB 또는 `run()`<br/>호출에 사용할 기본 정밀도입니다.<br/>각 입력 PUB는 자체 정밀도를 지정할 수 있습니다. `run()` 메서드에 정밀도가 주어지면, 그 값은 자체 정밀도를 지정하지 않은 `run()` 호출의 모든 PUB에 사용됩니다.  | float > 0 | 0.015625 |
| max_execution_time |  |  | 최대 실행 시간(초)으로, 벽시계 시간이 아닌 QPU 사용량을 기준으로 합니다. QPU 사용량은 QPU가 작업 처리에 전용으로 사용되는 시간입니다. 작업이 이 시간 제한을 초과하면 강제로 취소됩니다. | [1, 10800] 범위의 정수(초) |  |
| mitigation_level |  |  | 적용할 오류 억제 및 완화의 수준입니다. 각 수준에서 사용되는 방법에 대한 자세한 내용은 [완화 수준](#mitigation-level) 섹션을 참조하세요. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Circuit에 수행할 최적화의 양입니다. [높은 수준](/guides/set-optimization)은 더 최적화된 Circuit을 생성하지만 트랜스파일 시간이 더 오래 걸립니다. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | 동적 디커플링 활성화 여부입니다. 방법 설명은 [오류 억제 및 완화 기법](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling)을 참조하세요.  | True/False | True |
|  | sequence_type |  | 사용할 동적 디커플링 시퀀스입니다.<br/>* `XX`: `tau/2 - (+X) - tau - (+X) - tau/2` 시퀀스 사용<br/>* `XpXm`: `tau/2 - (+X) - tau - (-X) - tau/2` 시퀀스 사용<br/>* `XY4`: `tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` 시퀀스 사용 | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | 2-Qubit Clifford Gate 트위를링 적용 여부입니다. | True/False | False |
|  | enable_measure |  | 측정 트위를링 활성화 여부입니다. | True/False | True |
| resilience | measure_mitigation |  | TREX 측정 오류 완화 방법 활성화 여부입니다. 방법 설명은 [오류 억제 및 완화 기법](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex)을 참조하세요.  | True/False | True |
|  | zne_mitigation |  | Zero Noise Extrapolation 오류 완화 방법 활성화 여부입니다. 방법 설명은 [오류 억제 및 완화 기법](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne)을 참조하세요.  | True/False | False |
|  | zne | amplifier | 노이즈 증폭에 사용할 기법입니다. 다음 중 하나를 선택합니다. <br/> - `gate_folding`(기본값): 2-Qubit Gate 폴딩을 사용하여 노이즈를 증폭합니다. 노이즈 계수가 Gate의 일부만 증폭해야 하는 경우, 이 Gate들은 무작위로 선택됩니다.<br/><br/> - `gate_folding_front`: 2-Qubit Gate 폴딩을 사용하여 노이즈를 증폭합니다. 노이즈 계수가 Gate의 일부만 증폭해야 하는 경우, 이 Gate들은 위상 정렬된 DAG Circuit의 앞쪽에서 선택됩니다.<br/><br/> - `gate_folding_back`: 2-Qubit Gate 폴딩을 사용하여 노이즈를 증폭합니다. 노이즈 계수가 Gate의 일부만 증폭해야 하는 경우, 이 Gate들은 위상 정렬된 DAG Circuit의 뒤쪽에서 선택됩니다.<br/><br/> - `pea`: Probabilistic Error Amplification(PEA)이라는 기법을 사용하여 노이즈를 증폭합니다. 방법 설명은 [오류 억제 및 완화 기법](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea)을 참조하세요.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | 노이즈 증폭에 사용할 노이즈 계수입니다. | float 목록; 각 float >= 1 | PEA의 경우 (1, 1.5, 2), 그 외의 경우 (1, 3, 5). |
|  |  | extrapolator | 피팅 외삽 모델을 평가할 노이즈 계수입니다. 이 옵션은 실행 또는 모델 피팅에 영향을 주지 않으며, `extrapolator` 객체가 평가되어 `evs_extrapolated` 및 `stds_extrapolated` 데이터 필드로 반환되는 지점만 결정합니다. | `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)` 중 하나 이상 | (`exponential`, `linear`) |
|  | pec_mitigation |  | Probabilistic Error Cancellation 오류 완화 방법 활성화 여부입니다. 방법 설명은 [오류 억제 및 완화 기법](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)을 참조하세요.  | True/False | False |
|  | pec | max_overhead | 허용되는 최대 Circuit 샘플링 오버헤드이며, `None`은 최대값 없음을 의미합니다. | None / 1보다 큰 정수 | 100 |

다음 예제에서는 완화 수준을 1로 설정하면 초기에 ZNE 완화가 꺼지지만, `zne_mitigation`을 `True`로 설정하면 `mitigation_level`의 관련 설정이 재정의됩니다.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## 출력
Circuit function의 출력은 두 개의 필드를 포함하는 [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)입니다.

- 하나 이상의 [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) 객체. 이 객체는 `PrimitiveResult`에서 직접 인덱싱할 수 있습니다.

- 작업 수준 메타데이터.

각 `PubResult`에는 `data`와 `metadata` 필드가 포함됩니다.

- `data` 필드에는 기댓값 배열(`PubResult.data.evs`)과 표준 오류 배열(`PubResult.data.stds`)이 최소한 포함됩니다. 사용된 옵션에 따라 더 많은 데이터가 포함될 수 있습니다.

- `metadata` 필드에는 PUB 수준 메타데이터(`PubResult.metadata`)가 포함됩니다.

다음 코드 스니펫은 `PrimitiveResult`(및 관련 `PubResult`) 형식을 설명합니다.